# ② Read a ciphertext, or test it against many cipher keys

This notebook runs a transcribed ciphertext through trained models, either from the **shared
collection** of known keys or your own model files, and ranks which keys it most likely fits.
For each model it reports:

- how many of your symbols exist in that key,
- how confident the model is (compared with its own references),
- which language the decoded text reads most like.

The best result, or the best few if their scores are close, is downloaded automatically.
**Always read the outputs yourself:** the scores are a rough guide that has not yet been validated.

Your transcription must follow the [transcription format](https://github.com/mbruton0426/keybridge/blob/main/docs/TRANSCRIPTION_FORMAT.md): one manuscript
line per line, the cipher symbols separated by spaces and written exactly as in the keys, and a blank
symbol (word divider) written as `_`. **Digit ciphers** can be written as running digits: models for digit keys
read digit by digit and find where each code starts ([details](https://github.com/mbruton0426/keybridge/blob/main/docs/TRANSCRIPTION_FORMAT.md#digit-ciphers-running-digits)).

The guide explains the settings and how to read the results: [https://github.com/mbruton0426/keybridge/blob/main/docs/READING_AND_SCANNING_GUIDE.md](https://github.com/mbruton0426/keybridge/blob/main/docs/READING_AND_SCANNING_GUIDE.md)

**How to run:** fill in step ① and choose *Runtime → Run all*. A GPU is not needed.

*Privacy: your transcription is processed on Google's servers for this session only. To keep it on
your own computer, use the command-line version instead (see the guide).*

In [ ]:
#@title ① Your settings  { display-mode: "form" }

#@markdown ### Your transcription
#@markdown Leave empty to get an upload button.
TRANSCRIPTION_FILE = ""  #@param {type:"string"}
#@markdown **Standard:** *symbols separated by spaces*. *One character per symbol* is also accepted and converted (a blank symbol becomes `_`).
TRANSCRIPTION_FORMAT = "symbols separated by spaces"  #@param ["symbols separated by spaces", "one character per symbol"]
SKIP_LINES_STARTING_WITH = "##"  #@param {type:"string"}
#@markdown Digit models only: treat spaces as code boundaries. By default they are ignored, so running digits work and the model finds the codes.
USE_SPACES_AS_CODE_BOUNDARIES = False  #@param {type:"boolean"}

#@markdown ### Which models to test
WHICH_MODELS = "All shared models"  #@param ["All shared models", "Shared models with digit codes", "Shared models with single-character symbols", "Shared models with named symbols", "Specific shared models (IDs below)", "Only my own model file(s)"]
#@markdown Model IDs, comma-separated (only for "Specific shared models"):
MODEL_IDS = ""  #@param {type:"string"}
#@markdown Also test model files of your own (upload when asked, or give paths):
USE_MY_OWN_MODEL_FILES = False  #@param {type:"boolean"}
MY_MODEL_FILES = ""  #@param {type:"string"}

#@markdown ### Ranking
#@markdown Languages to consider, as codes (e.g. `de, la, it`). Empty means all available.
LANGUAGES = ""  #@param {type:"string"}
#@markdown Models scoring within this margin of the best are all treated as top results.
CLOSE_MARGIN = 0.05  #@param {type:"slider", min:0, max:0.3, step:0.01}
#@markdown Models whose key contains fewer than this share of your symbols are skipped. 0 tests every model.
MIN_SYMBOL_COVERAGE = 0.75  #@param {type:"slider", min:0, max:1, step:0.05}

#@markdown ### Results
AUTO_DOWNLOAD_TOP_RESULTS = True  #@param {type:"boolean"}
WRITE_DECIPHERMENTS_FOR_ALL_MODELS = False  #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE = False  #@param {type:"boolean"}
#@markdown *Advanced:* another model collection (Hugging Face repository or folder). Empty means the shared collection.
MODEL_COLLECTION = ""  #@param {type:"string"}
print("Settings saved.")

In [ ]:
#@title ② Prepare the notebook (no need to open)  { display-mode: "form" }
# Installs the decipherment package from GitHub. To test before the repository exists,
# upload the repository folder to Colab and set INSTALL_FROM to its path, e.g. "/content/my_repo".
INSTALL_FROM = "keybridge"

import os, sys, subprocess, importlib
try:
    import keybridge
except ImportError:
    target = INSTALL_FROM
    print("Installing (about a minute the first time)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", target], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit("Installation failed. Check your internet connection and try again.")
    importlib.invalidate_caches()
    import keybridge

import torch
from keybridge.colab import in_colab, find_or_upload, zip_files, offer_download, copy_to_drive
from keybridge.common import StopHere, stop, safe_name, SPACE_SEPARATED, ONE_CHAR
from keybridge import config

IN_COLAB = in_colab()
WORK = "/content" if IN_COLAB else os.getcwd()
UPLOAD_DIR = os.path.join(WORK, "uploads")
print(f"Ready (package version {keybridge.__version__}).")
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

In [ ]:
#@title ③ Optional: custom edit for Python users  { display-mode: "form" }
def custom_edit_transcription_line(symbols):
    """One transcription line as a list of symbols. Return the edited list. Leave unchanged to change nothing."""
    return symbols
print("Custom edit loaded.")

In [ ]:
#@title ④ Get your files  { display-mode: "form" }
from keybridge.transcription import read_transcription, parse_skip_prefixes
transcription_path = find_or_upload(TRANSCRIPTION_FILE, "transcription (.txt)", UPLOAD_DIR)
lines = read_transcription(transcription_path, TRANSCRIPTION_FORMAT, parse_skip_prefixes(SKIP_LINES_STARTING_WITH),
                           custom_edit_transcription_line, max_length=50)
n_symbols = sum(len(s) for _, s in lines)
print(f"Transcription: {os.path.basename(transcription_path)}, {len(lines)} lines, {n_symbols} symbols, "
      f"{len({x for _, s in lines for x in s})} distinct symbols.")
n_digits = sum(ch.isdigit() for _, s in lines for x in s for ch in x)
if n_digits and n_symbols and n_digits > 1.2 * n_symbols:
    print(f"  ({n_digits} digits: models for digit keys read these one digit at a time and find the codes themselves.)")
own_models = []
if USE_MY_OWN_MODEL_FILES or WHICH_MODELS.startswith("Only"):
    own_models = find_or_upload(MY_MODEL_FILES, "model file(s) (.pt)", UPLOAD_DIR, multiple=True)
    print("Your model files:", ", ".join(os.path.basename(p) for p in own_models))

In [ ]:
#@title ⑤ Find the models to test  { display-mode: "form" }
from keybridge.registry import ModelRegistry
registry, entries = None, []
types = {"Shared models with digit codes": "digits", "Shared models with single-character symbols": "characters",
         "Shared models with named symbols": "named"}
if not WHICH_MODELS.startswith("Only"):
    registry = ModelRegistry(MODEL_COLLECTION.strip() or None)
    if WHICH_MODELS.startswith("Specific"):
        entries = registry.entries(ids=[m.strip() for m in MODEL_IDS.split(",") if m.strip()])
    else:
        entries = registry.entries(symbol_type=types.get(WHICH_MODELS))
    total = len(registry.entries())
    print(f"The collection has {total} model(s); {len(entries)} match your choice.")
if not entries and not own_models:
    stop("There are no models to test. Check your choice in step ①, or test your own model file.")

In [ ]:
#@title ⑥ Test the transcription against the models  { display-mode: "form" }
from keybridge.scan import scan, top_group, describe_ranking
langs = {c.strip() for c in LANGUAGES.split(",") if c.strip()} or None
print(f"Testing {len(entries) + len(own_models)} model(s)...")
results = scan(lines, TRANSCRIPTION_FORMAT, registry, entries, own_models, langs, MIN_SYMBOL_COVERAGE,
               use_separators=USE_SPACES_AS_CODE_BOUNDARIES)
top = top_group(results, CLOSE_MARGIN)
print()
describe_ranking(results, top)

In [ ]:
#@title ⑦ Look at the best results  { display-mode: "form" }
from keybridge.data import readable
if not top:
    print("No model produced a reading (all were skipped). See the ranking above.")
for r in top:
    langs_txt = ", ".join(f"{name} {p:.2f}" for _, name, p in r.get("languages", [])) or "no language scores"
    print(f"=== {r['id']}  (score {r['score']:.2f}; reads like: {langs_txt})")
    for rd in r["readings"][:8]:
        print(f"  [{rd['line']:>4}] {readable(rd['units'])}")
    print()

In [ ]:
#@title ⑧ Save and download  { display-mode: "form" }
from keybridge.scan import write_scan_outputs
SAFE_NAME = safe_name(os.path.splitext(os.path.basename(transcription_path))[0])
OUT_DIR = os.path.join(WORK, "outputs", f"scan_{SAFE_NAME}")
all_files, top_files = write_scan_outputs(OUT_DIR, SAFE_NAME, results, top, os.path.basename(transcription_path),
                                          WRITE_DECIPHERMENTS_FOR_ALL_MODELS)
print("Saved:", *[os.path.basename(f) for f in all_files], sep="\n  ")
all_zip = zip_files(all_files, os.path.join(WORK, f"scan_{SAFE_NAME}_all_results.zip"), f"scan_{SAFE_NAME}")
if SAVE_TO_GOOGLE_DRIVE:
    copy_to_drive(OUT_DIR, f"cipher_scans/{SAFE_NAME}")
if AUTO_DOWNLOAD_TOP_RESULTS:
    top_zip = zip_files(top_files, os.path.join(WORK, f"scan_{SAFE_NAME}_top_results.zip"), f"scan_{SAFE_NAME}")
    offer_download(top_zip)
print(f"\nAll results (every model) are in {os.path.basename(all_zip)}; to get it, open the folder icon on the left.")